In [1]:
# 初始化模拟的设备配置信息
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import os
# 初始化 OpenAI Embeddings 或其他嵌入函数
embeddings = OpenAIEmbeddings()
BASE_DIR = os.getcwd()
print(BASE_DIR)
persist_directory = rf"{BASE_DIR}\common\VectorStore"
print(persist_directory)
# 创建 Chroma 对象
vector_store = Chroma(
    collection_name="vector_collection_for_agent",
    embedding_function=embeddings,
    persist_directory=persist_directory,
    collection_metadata={
        "vs_name":"test"
    }
)

D:\DevelopFiles\AIGC\HomeAutoAgent
D:\DevelopFiles\AIGC\HomeAutoAgent\common\VectorStore


In [5]:
import os
from langchain_core.documents import Document
import json
BASE_DIR = os.getcwd()
# 文件路径
file_path = rf"{BASE_DIR}\oneNetConfig.json"



# 打开并读取 JSON 文件内容
with open(file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)
data_documents:list[Document]=[]
# 遍历每个设备的 JSON 对象并打印
for device in data:
    dict_str=json.dumps(device, indent=2, ensure_ascii=False)
    
    # print(type(dict_str))
    data_documents.append(Document(page_content=dict_str))
ids=vector_store.add_documents(data_documents)
print(ids)

['49445fd6-8331-4ac3-9fe2-d0d931a6375b', '68dadea2-e51e-491e-a80a-f62dbbd618c6', '0c7bb8f4-5155-45a1-b07f-12b006bbd1ef', 'a6643775-ec34-4049-8693-739eaf62c35f', '4891288f-b9dc-4fe9-8b61-1250ba0b37fe', '920da463-0a10-423f-91bc-d98e37029922', '90f5e92f-0c2c-4b08-811a-a01fe95a9d18', '051b828b-0130-4208-9d5a-d8531a250c00', 'f8cad18e-63fb-4760-af79-b8061e7b2474']


In [6]:
from langchain_core.documents import Document
all=vector_store.get(
    # ids=['688875e8-b0d9-457f-ac82-68a78a4a06b4', '6803132b-29fb-46d3-9b03-7fde31994fc1', '3571348a-0d14-4231-abfa-a9684299c9d0', '4e812df8-bdd6-4371-8fed-664f231732ad', 'c7425539-357a-42b0-a6a5-0ad8bdfddccd', '1275019f-cbd4-476a-b453-df3a3b4570a3', '8415ad06-9114-4405-a222-5cc09a302451', '2bdf1514-9a6c-4663-a9f0-24a763c5cb78', '8b87e417-222e-43d9-a2f8-065e23005cfe']
)
# print(all)
for i in all["documents"]:
    doc=Document(page_content=i)
    print("="*8)
    print(doc)

page_content='{
  "product_id": {
    "type": "string",
    "description": "产品ID",
    "value": "mD97Vya1hi"
  },
  "device_name": {
    "type": "string",
    "description": "设备名称",
    "value": "d1"
  },
  "device_type": "LED Light",
  "params": {
    "type": "object",
    "description": "控制参数，字典格式",
    "properties": {
      "led": {
        "type": "boolean",
        "description": "LED开关状态",
        "value_range": "true/false"
      }
    }
  }
}'
page_content='{
  "product_id": {
    "type": "string",
    "description": "产品ID",
    "value": "aX23Jrf5xy"
  },
  "device_name": {
    "type": "string",
    "description": "设备名称",
    "value": "空调"
  },
  "device_type": "Air Conditioner",
  "params": {
    "type": "object",
    "description": "控制参数，字典格式",
    "properties": {
      "power": {
        "type": "boolean",
        "description": "开关状态",
        "value_range": "true/false"
      },
      "temperature": {
        "type": "integer",
        "description": "设定温度",
        "value

In [4]:

ids=vector_store.get()['ids']
vector_store.delete(ids=ids)

In [7]:
from langchain_core.messages import HumanMessage
from common.structs import DeviceModelFactory, DeviceCalls
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document

configs=[Document(page_content="""{
  "product_id": {
    "type": "string",
    "description": "产品ID",
    "value": "mD97Vya1hi"
  },
  "device_name": {
    "type": "string",
    "description": "设备名称",
    "value": "d1"
  },
  "device_type": "LED Light",
  "params": {
    "type": "object",
    "description": "控制参数，字典格式",
    "properties": {
      "led": {
        "type": "boolean",
        "description": "LED开关状态",
        "value_range": "true/false"
      }
    }
  }
}""")]

factory = DeviceModelFactory()
factory.generate_all(configs)
ConfigUnion = factory.get_union_type()
DeviceCallsDynamic = DeviceCalls[ConfigUnion]
DeviceCallsDynamic.__name__ = "DeviceCallsDynamic"
print(f"{DeviceCallsDynamic.__name__}+++++++++++++++++++++++++++++++++++++++++++++++++")

llm=ChatOpenAI(
    model="gpt-4o-mini"
)
llm=llm.with_structured_output(DeviceCallsDynamic)


llm.invoke([HumanMessage(content="请控制LED灯打开，参数随便")])

model_name:LEDLightConfig
DeviceCallsDynamic+++++++++++++++++++++++++++++++++++++++++++++++++


DeviceCallsDynamic(device_calls=[DeviceCall[LEDLightConfig](device_name='Living Room LED', device_id='led001', config=LEDLightConfig(led=True), order=1)])